# Unity Catalog: Managed vs External Tables

This notebook demonstrates creating **managed** and **external** Delta tables
using PySpark + Unity Catalog, with data stored on AWS S3.

## Key Differences

| | Managed Table | External Table |
|---|---|---|
| Storage | UC auto-assigns path under `s3://tuantm-data-platform` | You specify the LOCATION |
| DROP behavior | Deletes metadata **and** data | Deletes metadata only, data remains in S3 |
| Credentials | UC vends temp credentials to Spark | UC vends temp credentials to Spark |
| Use case | Tightly-coupled data lifecycle | Cross-system access, existing data |

```
┌────────────────┐
│  JupyterHub    │
│  (this notebook)│
└───────┬────────┘
        │ PySpark + UC Spark Connector
        ▼
┌────────────────┐         ┌─────────────────────────┐
│ Unity Catalog  │────────▶│      AWS S3              │
│  (metadata +   │ vends   │  s3://tuantm-data-platform│
│   credential   │ temp    │                           │
│   vending)     │ creds   │  managed/  ← UC controls │
│                │         │  external/ ← you control  │
└────────────────┘         └─────────────────────────┘
```

## Step 0: Check environment

In [ ]:
import os

UC_ENDPOINT = os.environ.get("UNITY_CATALOG_ENDPOINT", "http://server.unity-catalog.svc.cluster.local:8080")
print("UNITY_CATALOG_ENDPOINT:", UC_ENDPOINT)

# Paste your UC token here (obtained via get-uc-token.sh or token exchange)
UC_TOKEN = ""

## Step 1: Create catalog + schema via UC REST API

The Spark UC connector works with existing catalogs/schemas, so we create them first.

In [ ]:
import requests
import json

headers = {"Authorization": f"Bearer {UC_TOKEN}"}

# Check UC is reachable
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/catalogs", headers=headers)
print("UC catalogs response:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

In [ ]:
CATALOG_NAME = "tuantm"
SCHEMA_NAME = "demo"
STORAGE_ROOT = "s3://tuantm-data-platform/managed"

# Create catalog with managed storage root (requires CREATE CATALOG + CREATE_MANAGED_STORAGE)
resp = requests.post(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/catalogs",
    headers=headers,
    json={
        "name": CATALOG_NAME,
        "comment": "Demo catalog for tuantm",
        "storage_root": STORAGE_ROOT,
    },
)
if resp.status_code == 200:
    print(f"Created catalog: {CATALOG_NAME} (storage_root: {STORAGE_ROOT})")
elif resp.status_code == 409:
    print(f"Catalog already exists: {CATALOG_NAME}")
else:
    print(f"Create catalog response: {resp.status_code} - {resp.text}")

# Create schema
resp = requests.post(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/schemas",
    headers=headers,
    json={
        "name": SCHEMA_NAME,
        "catalog_name": CATALOG_NAME,
        "comment": "Demo schema",
    },
)
if resp.status_code == 200:
    print(f"Created schema: {CATALOG_NAME}.{SCHEMA_NAME}")
elif resp.status_code == 409:
    print(f"Schema already exists: {CATALOG_NAME}.{SCHEMA_NAME}")
else:
    print(f"Create schema response: {resp.status_code} - {resp.text}")

## Step 2: Create Spark session with Unity Catalog connector

Required packages:
- `unitycatalog-spark` — UC Spark connector (handles credential vending from UC)
- `delta-spark` — Delta Lake format
- `hadoop-aws` — S3A filesystem driver

**No AWS credentials needed in Spark config** — UC vends temporary credentials automatically.

In [ ]:
from pyspark.sql import SparkSession

# Stop existing Spark session if any
try:
    spark.stop()
except:
    pass

CATALOG_NAME = "tuantm"

spark = (
    SparkSession.builder
    .appName("UC-Managed-External-Tables-Demo")
    # --- Maven packages ---
    .config(
        "spark.jars.packages",
        ",".join([
            "io.unitycatalog:unitycatalog-spark_2.13:0.4.0",
            "io.delta:delta-spark_2.13:4.1.0",
            "org.apache.hadoop:hadoop-aws:3.4.2",
        ])
    )
    # --- Delta Lake extensions ---
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # --- Unity Catalog as the default catalog ---
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "io.unitycatalog.spark.UCSingleCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", UC_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.token", UC_TOKEN)
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # --- Local mode ---
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session created successfully!")

## Step 3: Verify catalog and schema

In [ ]:
spark.sql("SHOW CATALOGS").show()
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql("SHOW SCHEMAS").show()

---
## Step 4: MANAGED TABLE

A **managed table** is created **without** a `LOCATION` clause.
Unity Catalog:
- Auto-assigns a storage path under `s3://tuantm-data-platform/...`
- Vends temporary S3 credentials to Spark (no AWS keys needed in Spark config)
- **Deletes the data** when you DROP the table

In [ ]:
# MANAGED TABLE — no LOCATION specified, UC manages the storage
spark.sql("""
    CREATE TABLE IF NOT EXISTS tuantm.demo.users (
        id INT,
        name STRING,
        email STRING,
        created_at TIMESTAMP
    )
    USING delta
    TBLPROPERTIES ('delta.feature.catalogManaged' = 'supported');
""")

print("Managed table created: tuantm.demo.users")

In [ ]:
# Insert data into managed table
spark.sql("""
    INSERT INTO tuantm.demo.users VALUES
    (1, 'Tuan TM', 'tuantm@example.com', current_timestamp()),
    (2, 'Alice', 'alice@example.com', current_timestamp()),
    (3, 'Bob', 'bob@example.com', current_timestamp())
""")

print("Inserted 3 rows into managed table")

In [ ]:
spark.sql("SELECT * FROM tuantm.demo.users").show()

---
## Step 5: EXTERNAL TABLE

An **external table** is created **with** a `LOCATION` clause.
Unity Catalog:
- You control where data is stored (explicit S3 path)
- UC still vends temporary credentials for Spark to access S3
- **Does NOT delete data** when you DROP the table — only metadata is removed

In [ ]:
# EXTERNAL TABLE — explicit LOCATION on S3
spark.sql("""
    CREATE TABLE IF NOT EXISTS tuantm.demo.orders (
        order_id INT,
        user_id INT,
        product STRING,
        amount DOUBLE
    )
    USING delta
    LOCATION 's3://tuantm-data-platform/external/demo/orders'
""")

print("External table created: tuantm.demo.orders")
print("Data location: s3://tuantm-data-platform/external/demo/orders")

In [ ]:
# Insert data into external table
spark.sql("""
    INSERT INTO tuantm.demo.orders VALUES
    (101, 1, 'Laptop', 1200.0),
    (102, 2, 'Mouse', 25.0),
    (103, 1, 'Keyboard', 75.0),
    (104, 3, 'Monitor', 450.0),
    (105, 2, 'Headphones', 150.0)
""")

print("Inserted 5 rows into external table")

In [ ]:
spark.sql("SELECT * FROM tuantm.demo.orders").show()

---
## Step 6: Join managed + external tables

In [ ]:
spark.sql("""
    SELECT u.name, o.product, o.amount
    FROM tuantm.demo.users u
    JOIN tuantm.demo.orders o ON u.id = o.user_id
    ORDER BY o.amount DESC
""").show()

In [ ]:
# Aggregate: total spending per user
spark.sql("""
    SELECT u.name, COUNT(*) as num_orders, SUM(o.amount) as total_spent
    FROM tuantm.demo.users u
    JOIN tuantm.demo.orders o ON u.id = o.user_id
    GROUP BY u.name
    ORDER BY total_spent DESC
""").show()

---
## Step 7: Compare table types via UC REST API

Check the `table_type` field: `MANAGED` vs `EXTERNAL`

In [ ]:
resp = requests.get(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/tables",
    headers=headers,
    params={"catalog_name": "tuantm", "schema_name": "demo"}
)
tables = resp.json().get("tables", [])

for t in tables:
    print(f"Table: {t['name']}")
    print(f"  Type: {t.get('table_type', 'N/A')}")
    print(f"  Storage: {t.get('storage_location', 'N/A')}")
    print()

In [ ]:
# List all tables
spark.sql("SHOW TABLES IN tuantm.demo").show()

---
## Step 8: DataFrame API — create another external table

You can also use the DataFrame API to write tables.

In [ ]:
from pyspark.sql import Row

# Create sample products data
products_data = [
    Row(product_id=1, name="Laptop", category="Electronics", price=1200.0),
    Row(product_id=2, name="Mouse", category="Accessories", price=25.0),
    Row(product_id=3, name="Keyboard", category="Accessories", price=75.0),
    Row(product_id=4, name="Monitor", category="Electronics", price=450.0),
    Row(product_id=5, name="Headphones", category="Accessories", price=150.0),
]

products_df = spark.createDataFrame(products_data)

# Write as external table with explicit S3 path
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", "s3://tuantm-data-platform/external/demo/products") \
    .saveAsTable("tuantm.demo.products")

print("Created external table: tuantm.demo.products")

In [ ]:
spark.sql("SELECT * FROM tuantm.demo.products").show()

---
## Summary

### What we demonstrated

| Table | Type | Storage Location |
|---|---|---|
| `tuantm.demo.users` | **MANAGED** | Auto-assigned by UC under `s3://tuantm-data-platform/` |
| `tuantm.demo.orders` | **EXTERNAL** | `s3://tuantm-data-platform/external/demo/orders` |
| `tuantm.demo.products` | **EXTERNAL** | `s3://tuantm-data-platform/external/demo/products` |

### How credential vending works
1. Spark sends CREATE/READ/WRITE request to UC
2. UC checks its `server.properties` for matching `s3.bucketPath` credentials
3. UC vends **temporary** S3 credentials back to Spark
4. Spark uses those temp credentials to access S3 directly
5. No AWS keys needed in JupyterHub or Spark config!

### DROP behavior
- `DROP TABLE tuantm.demo.users` → deletes metadata **and** S3 data (managed)
- `DROP TABLE tuantm.demo.orders` → deletes metadata only, S3 data remains (external)